In [117]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [118]:
#Read dataset, print first 5 rows
df = pd.read_csv('data/credit_risk_dataset.csv')
print(df.head(5))
print(len(df))

   person_age  person_income person_home_ownership  person_emp_length  \
0          22          59000                  RENT              123.0   
1          21           9600                   OWN                5.0   
2          25           9600              MORTGAGE                1.0   
3          23          65500                  RENT                4.0   
4          24          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0    PERSONAL          D      35000          16.02            1   
1   EDUCATION          B       1000          11.14            0   
2     MEDICAL          C       5500          12.87            1   
3     MEDICAL          C      35000          15.23            1   
4     MEDICAL          C      35000          14.27            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  
0                 0.59                         Y                           3  


In [119]:
#Check for missing values in each column
df.isna().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

In [120]:
#check the proportion of young people among those with unknown years of employment length
#len(df[(df['person_emp_length'].isna()) & (df['person_age'] <= 25)]) / len(df[df['person_emp_length'].isna()])
#young people (<= 25) can be assumed to have no employment length, howerver, as the portion is only around 0.51, this assumption couldn't cover all the missing values. 
#Thus, we need to find other ways to fill in the missing values, or simply drop these rows. 

#check the maximum age of people with unknown years of employment length
#df[df['person_emp_length'].isna()]['person_age'].max()

In [121]:
#Y: default on file, N: no default on file
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'Y':1, 'N':0})

In [122]:
#define feature columns and target column
target = 'cb_person_default_on_file'
features = [c for c in df.columns if c != target]

#split the data into training and testing sets
X = df[features]
y = df[target]

In [123]:
cat_cols = [
    "person_home_ownership",
    "loan_intent",
    "loan_grade"
]
num_cols = df.columns.difference(cat_cols).tolist()
num_cols.remove('cb_person_default_on_file')

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [ ]:
#classifier of cb_person_default_on_file using RandomForestClassifier
classifier = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,          # 70% train, 30% temp
    random_state=42
    #stratify=y
)

X_vald, X_test, y_vald, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,          # 15% val, 15% test
    random_state=42
    #stratify=y_temp
)

classifier.fit(X_train, y_train)
#y_vald_pred = classifier.predict(X_vald)
y_vald_proba = classifier.predict_proba(X_vald)[:, 1]
#print(classification_report(y_test, y_pred))
#y_proba = classifier.predict_proba(X_test)[:, 1]
#print("ROC AUC:", roc_auc_score(y_test, y_proba))


4887 4887


In [146]:
def threshold_search(y_vald, y_vald_proba, beta=2):

    best_t, best_loss = None, float("inf")

    for t in np.linspace(0, 1, 1001):
        y_hat = (y_vald_proba > t).astype(int)
        cm = confusion_matrix(y_vald, y_hat)
        FP = cm[0,1]
        FN = cm[1,0]
        loss = FP + beta*FN
        if loss < best_loss:
            best_loss, threshold = loss, t
            best_FP, best_FN = FP, FN
            y_pred_threshold = y_hat
    return y_pred_threshold, threshold

y_pred_threshold, threshold = threshold_search(y_vald, y_vald_proba, beta=2)
print(classification_report(y_vald, y_pred_threshold))

              precision    recall  f1-score   support

           0       1.00      0.79      0.88      4016
           1       0.51      1.00      0.67       871

    accuracy                           0.83      4887
   macro avg       0.75      0.90      0.78      4887
weighted avg       0.91      0.83      0.85      4887



In [148]:
y_test_proba = classifier.predict_proba(X_test)[:, 1]
y_test_hat = (y_test_proba > threshold).astype(int)
print(classification_report(y_test, y_test_hat))

              precision    recall  f1-score   support

           0       1.00      0.79      0.88      4021
           1       0.51      1.00      0.67       867

    accuracy                           0.83      4888
   macro avg       0.75      0.89      0.78      4888
weighted avg       0.91      0.83      0.84      4888



In [149]:
param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

gs = GridSearchCV(
    estimator=classifier,      # your Pipeline
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

gs.fit(X_vald, y_vald)

print("Best C:", gs.best_params_["model__C"])
print("Best CV AUC:", gs.best_score_)
best_clf = gs.best_estimator_


Best C: 0.001
Best CV AUC: 0.9010007709185791


In [153]:
y_proba = best_clf.predict_proba(X_test)[:, 1]
print("Test AUC:", roc_auc_score(y_test, y_proba))

y_vald_proba = best_clf.predict_proba(X_vald)[:, 1]
y_pred_threshold, threshold = threshold_search(y_vald, y_vald_proba, beta=2)
y_test_proba = best_clf.predict_proba(X_test)[:, 1]
y_test_hat = (y_test_proba > threshold).astype(int)
print(classification_report(y_test, y_test_hat))

Test AUC: 0.8888419419730382
              precision    recall  f1-score   support

           0       0.99      0.78      0.87      4021
           1       0.49      0.97      0.65       867

    accuracy                           0.82      4888
   macro avg       0.74      0.88      0.76      4888
weighted avg       0.90      0.82      0.84      4888

